# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

## 1. My lane as an ML task (type)

My chosen lane is **Refresh / Content Opportunity Scoring**. This is fundamentally a **Scoring and Ranking** task. Under the hood, we are using a **binary classification** model to predict a probability (likelihood of decay). We then use that probability (combined with potential impact, like traffic volume) to score and rank pages so an editor can review them in priority order.

In [1]:
import pandas as pd
import os

data_path = '../../../flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)
print("Task framing: Classification -> Scoring -> Ranking")

Task framing: Classification -> Scoring -> Ranking


## 2. Target or proxy

The target is whether a page is currently experiencing traffic decay. In this dataset, this is captured by the observed `trend_direction` column (specifically, `trend_direction == 'down'`). This serves as our proxy for "needs a refresh." It's an observed outcome based on recent traffic performance, not a subjective human label.

In [2]:
# Create the binary target
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
print(f"Target defined. Class balance (declining vs not):\n{df['is_declining_label'].value_counts(normalize=True).round(3)}")

Target defined. Class balance (declining vs not):
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

Since this model feeds a ranked queue for human review, predicting the exact probability for every page isn't as important as getting the *top* of the list right. The success metric is **Precision@K** (e.g., Precision@50). This tells us: out of the top 50 pages the model recommends updating, what percentage are actually decaying? A high Precision@K means we are not wasting the editors' time.

In [3]:
import numpy as np

# A quick dummy simulation to show Precision@K logic
def precision_at_k(labels, k):
    return labels[:k].mean()

print("Success metric: Precision@K")
print("Good = of the top K recommendations, a high fraction are actually true 'down' trends.")

Success metric: Precision@K
Good = of the top K recommendations, a high fraction are actually true 'down' trends.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one page (URL) at a specific snapshot in time**. Each row represents a single piece of content and its historical performance metrics leading up to the current date.

In [4]:
print(f"Unit of analysis: 1 row = 1 page (anonymized hash)")
display(df[['client_id', 'content_id', 'content_age_days', 'impressions_90d', 'is_declining_label']].head(3))

Unit of analysis: 1 row = 1 page (anonymized hash)


,client_id,content_id,content_age_days,impressions_90d,is_declining_label
0,client_f369cb89fc,content_304f48230142,187,3803,1
1,client_4e07408562,content_a1fb4e703a9e,445,15320,1
2,client_7f2253d7e2,content_9aa793d4d895,141,12581,1


## 5. Why ML beats a fixed rule here

A simple fixed rule (e.g., "update if age > 180 days AND impressions > 500") is brittle. It treats a 181-day-old page with 501 impressions exactly the same as a 500-day-old page with 50,000 impressions. ML can capture non-linear relationships and interactions between features—like how the impact of `content_age_days` changes depending on `avg_position` and `ctr`—finding the "messy" boundary between pages that are safely stable and those genuinely decaying.

In [5]:
# Showing the messy overlap: Even declining pages have varying ages and impression counts.
subset = df[df['is_declining_label'] == 1]
print("Summary of declining pages (showing variance that rules miss):")
print(subset[['content_age_days', 'impressions_90d']].describe().loc[['min', '25%', '50%', '75%', 'max']].round(0))

Summary of declining pages (showing variance that rules miss):
     content_age_days  impressions_90d
min              90.0              1.0
25%             126.0            179.0
50%             216.0            961.0
75%             313.0           3832.0
max             557.0         517715.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.